# 实战：牛仔服饰目标检测（YOLOv5）

本节把《动手学深度学习》目标检测一章的思路用到真实比赛上：Kaggle 牛仔服饰检测
（CodaLab 33573），数据是 COCO 格式的 5 类牛仔穿搭标注
（belt / sunglasses / boot / cowboy_hat / jacket）。与书里“从零实现 TinySSD”不同，
实战走迁移学习路线：用 ultralytics 官方维护的 **YOLOv5su**（YOLOv5-s 换上 v8
anchor-free 检测头的升级版，简称 YOLOv5u）加载 COCO 预训练权重，在 3000 多张图上微调。

**为什么选 YOLOv5**：单阶段检测器训练比两阶段的 Faster R-CNN 快得多；P3 浅层检测头 +
大输入分辨率（imgsz=1024）对小目标更友好（belt 在低分辨率下只剩几十个像素）；
mosaic 四图拼接增强对稀有类也很有帮助。

**本机环境注意事项**（书里不需要，本机必须）：

1. `vendor/` 下放了一份完好的 numpy 1.26.4，在任何 numpy 被 import 之前用
   `sys.path.insert(0, ...)` 优先加载（本机 myenv 的 numpy 需要这个垫片）；
2. 换 YOLOv5 只需 `pip install ultralytics`（唯一新增安装，不动 numpy/torch 等已有包）；
3. 预训练权重 `yolov5su.pt` 已放在项目根目录；丢失可从镜像手动下：
   `https://ghproxy.net/https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov5su.pt`

流程与书里检测实战章节一一对应：读取数据集 → 转换成 YOLO 标签格式 → 训练模型 →
评估模型（COCO 官方 mAP）→ 提交预测结果。

In [1]:
# 环境初始化：路径定位 + vendor numpy 注入 + CUDA 探测 + 导入
import os, sys, json, zipfile, random, logging, gc
from collections import Counter

# ---- 定位项目根目录（notebook 就在数据文件夹里）----
PROJECT_ROOT = os.getcwd()
if not os.path.exists(os.path.join(PROJECT_ROOT, 'train.json')):
    alt = os.path.join(os.path.dirname(PROJECT_ROOT), 'CowBoy Outfits Detection')
    if os.path.exists(os.path.join(alt, 'train.json')):
        PROJECT_ROOT = alt
os.chdir(PROJECT_ROOT)  # 后面所有相对路径都以它为基准

# ---- vendor numpy 注入：必须在任何 numpy import 之前（本机环境特有，书里不需要）----
# 本机 myenv 自带的 numpy 编译文件损坏，把一份完好的 numpy 1.26.4 放在 vendor/，
# 用 sys.path.insert(0, ...) 让它抢先被加载（包括被 torch/pandas/d2l 间接导入时）
VENDOR = os.path.join(PROJECT_ROOT, 'vendor')
if os.path.isdir(os.path.join(VENDOR, 'numpy')):
    sys.path.insert(0, VENDOR)

import numpy as np
import torch

# CUDA 探测必须在 import torchvision 之前（本机先加载 torchvision 的 CUDA ops
# 再探测会段错误）。注意 d2l/ultralytics 导入都会连带 import torchvision
HAS_CUDA = torch.cuda.is_available()

# d2l 工具库：try_gpu/Timer/show_bboxes 等全书通用函数都走 d2l.xxx 命名空间
from d2l import torch as d2l
device = d2l.try_gpu()  # 永不硬编码设备

# YOLOv5 走 ultralytics 包：自带数据加载/增强/训练循环；评估用 pycocotools
import ultralytics
from ultralytics import YOLO
from pycocotools.coco import COCO          # 解析 COCO 格式标注 train.json
from pycocotools.cocoeval import COCOeval  # 官方 mAP 评估器
import pandas as pd
from PIL import Image

# 关闭 ultralytics 的联网遥测（大陆网络下省得做无谓的联网尝试）
try:
    from ultralytics import settings as _yolo_settings
    _yolo_settings.update({'sync': False})
except Exception:
    pass

# 防御性检查：确认 numpy 确实是从 vendor/ 加载的，否则说明注入失败
assert 'vendor' in np.__file__, 'numpy 应来自 vendor/，检查 sys.path 注入'

# 固定随机种子，保证数据划分与训练可复现
torch.manual_seed(0); np.random.seed(0); random.seed(0)

# 日志输出到 stderr（pycocotools 会往 logger 里写评估信息）
logger = logging.getLogger()
logger.addHandler(logging.StreamHandler(sys.stderr))

# 冒烟测试开关：设环境变量 COWBOY_QUICK=1 后用小数据 + 少量 epoch 快速验证流程
QUICK = os.environ.get('COWBOY_QUICK') == '1'

print('numpy       ', np.__version__, '@', os.path.dirname(np.__file__))
print('torch       ', torch.__version__, '| cuda:', HAS_CUDA)
print('ultralytics ', ultralytics.__version__)
print('device      ', device)
print('root        ', PROJECT_ROOT)

torch        2.13.0+cu132 | cuda: True
ultralytics  8.4.143
device       cuda:0


## 读取数据集

数据集来自 GluonCV 团队制作的 CowboySuit（COCO 格式）：`train.json` 存全部标注，
`images/` 存图片。先解析标注建立类别映射，再把标注整理成每张图一行的表格。

In [ ]:
# 读取 COCO 格式标注，建立类别映射
coco = COCO(os.path.join(PROJECT_ROOT, 'train.json'))
print('Data info:', coco.dataset.get('info'))

# {类别名: COCO类别id}。注意 COCO 的类别 id 是任意整数（不一定从 1 连续），
# 所以训练一律用连续小整数（YOLO 索引 0~N-1），输出结果时再查表换回 COCO id
categories = {cat['name']: cat['id'] for cat in coco.loadCats(coco.getCatIds())}
print('Categories:', categories)

class_names = list(categories.keys())                                          # 顺序即类别索引的依据
# YOLO 的类别索引是 0~N-1（顺序 = class_names），预测结果换回比赛 id 时查这张表
yolo_cls_to_cocoid = {i: categories[n] for i, n in enumerate(class_names)}
print('class_names:', class_names)
print('yolo_cls_to_cocoid:', yolo_cls_to_cocoid)

In [3]:
# 把所有标注整理成 DataFrame：每张图片一行，rois 列存该图所有目标框
def load_rois(coco, img):
    """取出一张图片的全部标注框，转成归一化坐标（0~1）的字典列表"""
    anns = coco.loadAnns(coco.getAnnIds(imgIds=img['id']))  # 按图片 id 查它所有标注
    rois = []
    for a in anns:
        x, y, w, h = a['bbox']  # COCO 的框是 [左上x, 左上y, 宽, 高]（像素）
        rois.append({'class': next(n for n, cid in categories.items() if cid == a['category_id']),
                     'xmin': x / img['width'],  'ymin': y / img['height'],
                     'xmax': (x + w) / img['width'], 'ymax': (y + h) / img['height'],
                     'area': a['area'],                 # 框面积，COCO 评估区分大中小目标时用
                     'iscrowd': a.get('iscrowd', 0)})   # 密集人群标注，训练时跳过
    return rois


rows = [{'image': os.path.join(PROJECT_ROOT, 'images', img['file_name']),
         'image_id': img['id'], 'width': img['width'], 'height': img['height'],
         'rois': load_rois(coco, img)} for img in coco.dataset['images']]
train = pd.DataFrame(rows)
print('total images:', len(train))

# 类别分布严重不均衡：jacket 2195 / sunglasses 2330 / cowboy_hat 595 / boot 449 / belt 25
cnt = Counter()
for rois in train['rois']:
    for r in rois:
        cnt[r['class']] += 1
print('class distribution:', dict(cnt))

# ---- 按类别分层切分 train/valid：每个类别都抽几张含该类的图进验证集，保证
# 验证集里每个类别都有样本（一张图可含多类，需去重）。稀有类要封顶：belt 全集
# 只有 21 张图，也抽 20 张的话训练集里 belt 就一张不剩了，所以每类最多抽
# 「该类图片数的 1/5」（下限 2 张、上限 20 张）----
sample_n_per_cat = 20
valid_ids = []
for cat_name in categories:
    mask = train['rois'].apply(lambda rois: any(r['class'] == cat_name for r in rois))
    df = train[mask]
    n_valid = min(sample_n_per_cat, max(2, len(df) // 5), len(df))  # 稀有类少留验证、多留训练
    df = df.sample(n_valid, random_state=0)
    valid_ids += list(df.index)
valid_ids = list(set(valid_ids))
train_ids = [i for i in train.index if i not in valid_ids]
train_data = train.loc[train_ids]
valid_data = train.loc[valid_ids]
print('train split:', len(train_data), 'valid split:', len(valid_data))

# ---- belt 过采样：按类别平均的 mAP 里 25 个框和 2330 个框话语权一样（各占 1/5），
# 把含 belt 的训练图复制几份（复制在图片列表里实现，见下一节）。配合随机翻转和
# mosaic 增强，同一张图每个 epoch 看到的都不同，重复不等于死记硬背 ----
BELT_COPIES = 6  # 过采样后 belt 图的总份数（1 份原始 + 5 份重复）
belt_mask = train_data['rois'].apply(lambda rois: any(r['class'] == 'belt' for r in rois))
print('belt images in train:', int(belt_mask.sum()))
if belt_mask.sum() > 0:
    train_data = pd.concat([train_data] + [train_data[belt_mask]] * (BELT_COPIES - 1),
                           ignore_index=True)
    print(f'belt oversample x{BELT_COPIES} -> train split: {len(train_data)}')

if QUICK:  # 冒烟测试：压缩数据量，只验证流程能跑通
    train_data = train_data.sample(min(64, len(train_data)), random_state=0)
    valid_data = valid_data.sample(min(20, len(valid_data)), random_state=0)
    print('quick mode -> train:', len(train_data), 'valid:', len(valid_data))

total images: 3062
class distribution: {'jacket': 2195, 'boot': 449, 'sunglasses': 2330, 'cowboy_hat': 595, 'belt': 25}
train split: 2978 valid split: 84
belt images in train: 17
belt oversample x6 -> train split: 3063


## 转换成 YOLO 标签格式

ultralytics 的训练接口不吃 COCO json，而是吃 YOLO 格式：每张图对应一个同名 txt
标签（每行 `类别索引 中心x 中心y 宽 高`，坐标按图像宽高归一化到 0~1），外加一个
指明图片列表和类别名的 yaml 配置。这一节把 COCO 标注转换过去——并且不复制任何图片。

In [4]:
# YOLO 标签格式：labels/*.txt + 图片列表 train_yolo.txt / val_yolo.txt + cowboy.yaml
LABEL_DIR = os.path.join(PROJECT_ROOT, 'labels')
os.makedirs(LABEL_DIR, exist_ok=True)

#@save
def yolo_label_path(image_path):
    """图片路径 -> YOLO 标签 txt 路径（images 段换成 labels，后缀换 .txt）"""
    base, _ = os.path.splitext(image_path)
    return base.replace(os.sep + 'images' + os.sep, os.sep + 'labels' + os.sep) + '.txt'

# 【免复制图片】ultralytics 由图片路径推标签路径的规则是：把路径中最后一段
# "images" 换成 "labels"（img2label_paths），所以 txt 放到与 images/ 同级的
# labels/ 目录、文件同名，再在图片列表里列出图片绝对路径即可。
# belt 过采样也顺便用列表实现：train_data 里被复制的行会在列表里多写几遍。
written = set()
for df in (train_data, valid_data):
    for _, row in df.iterrows():
        lines = []
        for r in row['rois']:
            if r['iscrowd']:  # 密集人群标注框质量差，跳过
                continue
            cls_idx = class_names.index(r['class'])   # YOLO 类别索引 0~4
            cx = (r['xmin'] + r['xmax']) / 2          # 归一化中心点
            cy = (r['ymin'] + r['ymax']) / 2
            w, h = r['xmax'] - r['xmin'], r['ymax'] - r['ymin']
            lines.append(f'{cls_idx} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
        # 空列表 = 纯背景图，ultralytics 允许
        lp = yolo_label_path(row['image'])
        with open(lp, 'w', encoding='utf-8') as f:
            f.write('\n'.join(lines))
        written.add(lp)
print('label files written:', len(written))

train_list = os.path.join(PROJECT_ROOT, 'train_yolo.txt')
val_list = os.path.join(PROJECT_ROOT, 'val_yolo.txt')
with open(train_list, 'w', encoding='utf-8') as f:
    f.write('\n'.join(train_data['image']) + '\n')
with open(val_list, 'w', encoding='utf-8') as f:
    f.write('\n'.join(valid_data['image']) + '\n')
print('train list:', len(train_data), 'rows | val list:', len(valid_data), 'rows')

# 数据集配置：nc 是类别数，names 顺序决定类别索引；
# 训练时 ultralytics 据此自动把检测头重建成 5 类（迁移学习一步到位）
DATA_YAML = os.path.join(PROJECT_ROOT, 'cowboy.yaml')
with open(DATA_YAML, 'w', encoding='utf-8') as f:
    f.write(f'train: {train_list}\nval: {val_list}\n'
            f'nc: {len(class_names)}\nnames: {class_names}\n')
print(open(DATA_YAML, encoding='utf-8').read())

label files written: 3062
train list: 3063 rows | val list: 84 rows
nc: 5
names: ['belt', 'sunglasses', 'boot', 'cowboy_hat', 'jacket']



## 训练模型

加载 COCO 预训练的 `yolov5su.pt`（约 18MB，已放在项目根目录；丢失可从镜像手动下：
`https://ghproxy.net/https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov5su.pt`）。
迁移学习在这里不需要手工换分类头——ultralytics 读到 yaml 里的 5 个类别会自动重建
检测头，其余主干/颈部权重全部保留微调。

In [ ]:
# 加载权重：优先用已训练的 best.pt（重跑评估/提交/可视化时无需重新训练）；
# 想从 COCO 预训练权重重新训练，删掉 yolov5_cowboy/ 目录即可
BEST = os.path.join(PROJECT_ROOT, 'yolov5_cowboy', 'weights', 'best.pt')
if os.path.exists(BEST):
    WEIGHTS = BEST
    print('发现已训练的 best.pt，直接加载（想重新训练请删除 yolov5_cowboy/）')
else:
    WEIGHTS = os.path.join(PROJECT_ROOT, 'yolov5su.pt')
    if not os.path.exists(WEIGHTS):
        print('本地没有 yolov5su.pt，尝试让 ultralytics 自动下载（github，可能超时）...')
        WEIGHTS = 'yolov5su.pt'  # 下载成功会落在 cwd；失败就按上一节的镜像地址手动下载

model = YOLO(WEIGHTS)

# 超参集中赋值：输入分辨率（letterbox 到正方形）是小目标的第一瓶颈，
# 8GB 显存 @1024 配 s 档正好；OOM 就降 768/640，或换 yolov5mu.pt（batch 相应减半）
imgsz, eval_imgsz = 1024, 1024

# 显存自检：kernel 里旧训练的残留不会自动释放（异常 traceback 会持有出错瞬间的
# 中间张量），OOM 之后不重启 kernel 的话后面每个 cell 都会继续 OOM
gc.collect(); torch.cuda.empty_cache()
free_gb = torch.cuda.mem_get_info()[0] / 2**30
print(f'GPU 空闲显存 {free_gb:.1f} GiB（分块推理约需 2，不足就 Kernel→Restart）')

print('model loaded | task:', model.task, '| device:', device)

In [ ]:
# 验证集推理 + COCO 官方 mAP 评估
# 【坑】ultralytics 的 predict 对「列表源」不分批：check_source 会把整个列表用
# autocast_list 一次性读进内存，LoadPilAndNumpy 直接取 bs=列表长度 —— 999 张图
# 就是 999 张一个 batch（首层卷积输出要 15.6GiB，必 OOM）。batch 参数只对
# 文件路径/目录/txt 源生效。所以这里自己按 PREDICT_BATCH 切块逐段推理。
PREDICT_BATCH = 16  # @1024+TTA 单块峰值约 1GB；显存富余可加大

#@save
def predict_paths(model, paths, imgsz=eval_imgsz, tta=True):
    """按 PREDICT_BATCH 分块对一批图片路径推理，返回 Results 列表（与 paths 一一对应）"""
    paths = list(paths)
    results = []
    for i in range(0, len(paths), PREDICT_BATCH):
        chunk = paths[i:i + PREDICT_BATCH]  # 切块保序，Results 顺序与 paths 一致
        # conf=0.05：评测按 AP 计分（每图最多取 top-100 检测），
        # 多保留低分框一般不掉分、反而可能提高召回
        # tta=True：内置测试时增强（水平翻转+多尺度），AP 通常能白拿零点几个点
        # OpenCV 读图自动转 3 通道，RGBA 图无需手工 convert('RGB')
        results += model.predict(chunk, imgsz=imgsz, conf=0.05, iou=0.7,
                                 augment=tta, half=HAS_CUDA, device=device,
                                 verbose=False)
    return results


#@save
def results_to_coco(results, path_to_id):
    """把 ultralytics Results 转成 COCO 结果格式（bbox 为 [x,y,w,h] 像素）"""
    out = []
    for res in results:
        img_id = path_to_id[res.path]  # 图片路径 -> 原始（大整数）image_id
        for b, cls, s in zip(res.boxes.xyxy.tolist(), res.boxes.cls.tolist(),
                             res.boxes.conf.tolist()):
            x1, y1, x2, y2 = b
            out.append({'image_id': img_id,
                        'category_id': yolo_cls_to_cocoid[int(cls)],  # 换回 COCO id 才能对上 GT
                        'bbox': [x1, y1, x2 - x1, y2 - y1],
                        'score': float(s)})
    return out


#@save
def evaluate_map(model, valid_df, imgsz=eval_imgsz, verbose=True):
    """用 pycocotools 的 COCOeval 计算 mAP@[.5:.95]（COCO 主指标）"""
    path_to_id = {row['image']: int(row['image_id']) for _, row in valid_df.iterrows()}
    results = results_to_coco(predict_paths(model, list(path_to_id), imgsz=imgsz), path_to_id)
    gt_ids = list(set(path_to_id.values()))

    # 【坑】image id 是 uint64，可能超过 int64 上限；COCOeval 内部静默转 float64
    # 丢精度，导致 GT 和检测结果匹配不上、mAP 恒为 -1。
    # 先把 id 映射成 0..N-1 的连续小整数，GT 和 DT 用同一套映射
    id2small = {uid: i for i, uid in enumerate(gt_ids)}
    gt = {'images': [{'id': id2small[im['id']],
                      'width': im['width'], 'height': im['height'],
                      'file_name': im['file_name']}
                     for im in coco.dataset['images'] if im['id'] in id2small],
          'annotations': [dict(a, image_id=id2small[a['image_id']])
                          for a in coco.dataset['annotations'] if a['image_id'] in id2small],
          'categories': coco.dataset['categories']}
    results = [dict(r, image_id=id2small[r['image_id']]) for r in results]

    # 本机 pycocotools 的 COCO() 只收文件路径，先落盘临时 json 再读入，用完即删
    import tempfile
    with tempfile.NamedTemporaryFile('w', suffix='.json', delete=False, encoding='utf-8') as f:
        json.dump(gt, f)
        gt_path = f.name
    try:
        gt_coco = COCO(gt_path)
    finally:
        os.remove(gt_path)

    dt_coco = gt_coco.loadRes(results)  # 把检测结果挂到 GT 上
    ev = COCOeval(gt_coco, dt_coco, 'bbox')
    ev.evaluate(); ev.accumulate()
    if verbose:
        # 按类别打印 AP@IoU=0.50，定位拖后腿的类（如稀有类 belt）
        # precision 形状 [IoU阈值, 召回点, 类别, 面积范围, maxDets]
        prec = ev.eval['precision']
        for k, cid in enumerate(ev.params.catIds):
            p = prec[:, :, k, 0, -1]  # 所有 IoU 阈值、area=all、maxDets=100
            ap50 = p[p > -1].mean() if (p > -1).any() else float('nan')
            print(f"    AP50[{coco.loadCats([cid])[0]['name']:>10s}] = {ap50:.3f}")
        ev.summarize()  # 打印 AP/AP50/AP75/大中小目标 AP 等完整指标
    return ev.stats[0]  # stats[0] 就是 mAP@[.5:.95]


# 版本标记：重跑本 cell 后 kernel 里的 predict_paths 才是分块版。
# 如果提交 cell 报 OOM 且没有这行输出，说明 kernel 里还是旧函数——重跑本 cell
print('predict_paths 就绪：分块版 PREDICT_BATCH =', PREDICT_BATCH)

In [7]:
# 超参集中赋值：YOLO 收敛快，小数据集想冲分可把 num_epochs 加到 30~50
# 探针实测 @1024 训练显存峰值：b4 2.0G / b6 3.0G / b8 3.9G（干净进程都能跑）。
# 默认取 4 是给 kernel 残留留余量——反复跑 cell / OOM 后不重启时 b8 会撞墙；
# 跑通一轮后想提速可提到 6/8（前提：刚重启的干净 kernel）
num_epochs, batch_size = (12,4)

# ultralytics 内置训练循环：mosaic+HSV+翻转增强、优化器自动配置、cos_lr 余弦退火；
# 每个 epoch 自带一次验证，并按验证 fitness 保存 weights/best.pt
timer = d2l.Timer()
timer.start()
model.train(data=DATA_YAML, epochs=num_epochs, imgsz=imgsz, batch=batch_size,
            device=device, workers=4,     # Windows 下卡死/报 pickle 错就改回 0
            cos_lr=True, close_mosaic=4,  # 最后 4 个 epoch 关 mosaic，适应真实分布
            patience=num_epochs + 1, seed=0,
            project=PROJECT_ROOT,         # 输出在 PROJECT_ROOT/yolov5_cowboy/（8.4 无 runs/detect 层）
            name='yolov5_cowboy', exist_ok=True,
            cache=False, verbose=True)
timer.stop()
# d2l 惯例的收尾打印：平均每秒处理多少张训练图
print(f'{len(train_data) * num_epochs / timer.sum():.1f} images/sec on {str(device)}')

# 加载 best 权重（按验证 fitness 挑的最优轮，不是最后一轮）用于评估和提交
save_dir = getattr(getattr(model, 'trainer', None), 'save_dir', None)
best_pt = os.path.join(str(save_dir), 'weights', 'best.pt') if save_dir \
    else os.path.join(PROJECT_ROOT, 'yolov5_cowboy', 'weights', 'best.pt')
assert os.path.exists(best_pt), f'best.pt 未找到: {best_pt}'
model = YOLO(best_pt)
print('loaded best:', best_pt)

# 清掉训练期的缓存显存块，给接下来的 TTA 评估腾地方
gc.collect(); torch.cuda.empty_cache()

# 用同一套 COCOeval 口径复评，结果与 Faster R-CNN 版直接可比
mAP = evaluate_map(model, valid_data)
print(f'valid mAP@[.5:.95] {mAP:.3f}')

New https://pypi.org/project/ultralytics/8.4.144 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.143  Python-3.11.15 torch-2.13.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8123MiB)
Overriding model.yaml nc=80 with nc=5

                   from  n    params  module                                       arguments                     
  0                  -1  1      3520  ultralytics.nn.modules.conv.Conv             [3, 32, 6, 2, 2]              
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     18816  ultralytics.nn.modules.block.C3              [64, 64, 1]                   
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    115712  ultralytics.nn.modules.block.C3              [128, 128, 2]                 
  5                  -1  1    295424  ultralytics.nn.modules.conv.Co

<MYENV>\Lib\site-packages\ray\train\_internal\session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/12      2.88G      1.746      2.241        1.8         10       1024: 100% ━━━━━━━━━━━━ 766/766 2.8it/s 4:320.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 12.4it/s 0.9s0.1s
                   all         84        155      0.554       0.27      0.284      0.134

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/12      2.88G      1.771      2.196      1.839          4       1024: 100% ━━━━━━━━━━━━ 766/766 2.8it/s 4:310.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 13.1it/s 0.8s0.1s
                   all         84        155       0.54      0.266      0.296      0.145

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/12      2.88G      1.742      2.145      1.821         12       1024: 100% ━━━━━━━━━━━━ 766/766 2.8it/s 4:300.3ss
                 Class     Imag

## 提交预测结果

`valid.csv` 用于公开榜提交，`test.csv` 用于决赛提交——决赛只有 3 次机会，别在
最后一天传错文件。比赛要求每行 `{image_id, category_id, bbox:[x1,y1,x2,y2], score}`，
注意 bbox 是左上/右下角格式，与 COCO 评估用的左上+宽高不同。

In [10]:
#@save
def create_submission(df, model, score_thresh=0.05, tta=True):
    """对 DataFrame 里的每张图推理，返回提交格式的字典列表"""
    results = []
    # predict_paths 内部按 PREDICT_BATCH 切块推理（列表源不分批，见其注释）；
    # RGBA 图由 OpenCV 自动转 3 通道
    preds = predict_paths(model, df['file_name'], tta=tta)
    for pred, (_, row) in zip(preds, df.iterrows()):
        img_id = int(row['id'])  # csv 里按字符串读入的大整数 id
        for b, cls, s in zip(pred.boxes.xyxy.tolist(), pred.boxes.cls.tolist(),
                             pred.boxes.conf.tolist()):
            if s > score_thresh:
                x1, y1, x2, y2 = b
                results.append({'image_id': img_id,
                                'category_id': yolo_cls_to_cocoid[int(cls)],
                                'bbox': [x1, y1, x2, y2],  # 角格式，别写成 [x,y,w,h]
                                'score': float(s)})
    return results

In [ ]:
# 生成 answer.json 并打包成 sample_answer.zip
SUBMIT_SPLIT = 'valid'  # 公开榜用 'valid'；决赛时改成 'test'（只有 3 次提交机会）

# id 是 uint64（可能超过 int64 上限），必须按字符串读入，用的时候再 int() 转回大整数
submission_df = pd.read_csv(os.path.join(PROJECT_ROOT, f'{SUBMIT_SPLIT}.csv'),
                            dtype={'id': 'object'})
submission_df['file_name'] = submission_df['file_name'].apply(
    lambda fn: os.path.join(PROJECT_ROOT, 'images', fn))

if QUICK:  # 冒烟测试只跑前 8 张，验证生成链路即可
    submission_df = submission_df.head(8)
    print('quick mode -> submission images:', len(submission_df))

gc.collect(); torch.cuda.empty_cache()  # 批量 TTA 推理前清一次缓存块

timer = d2l.Timer()
submission = create_submission(submission_df, model)
timer.stop()
print('predictions:', len(submission),
      f'({len(submission_df) / timer.sum():.1f} images/sec)')

# 写 answer.json 并打包：CodaLab 要求提交 zip 且内部文件名必须是 answer.json
answer_path = os.path.join(PROJECT_ROOT, 'answer.json')
with open(answer_path, 'w') as f:
    json.dump(submission, f)
zip_path = os.path.join(PROJECT_ROOT, 'sample_answer.zip')
with zipfile.ZipFile(zip_path, 'w') as zf:
    zf.write(answer_path, 'answer.json')  # 第二个参数指定 zip 内的文件名（不带目录）
print('saved:', answer_path, '|', zip_path)

## 提交到 CodaLab

把 `sample_answer.zip`（内含 `answer.json`）上传到
https://competitions.codalab.org/competitions/33573#participate-submit_results 。

In [ ]:
# 可视化：训练集标注示例 + 验证集预测效果（用书里的 bbox_to_rect/show_bboxes 工具）
%matplotlib inline

# 类别 -> 固定颜色：画 GT 与预测用同一套，便于对照
palette = ['red', 'blue', 'green', 'orange', 'purple']

def cls_color(name):
    """类别名 -> 固定颜色"""
    return palette[class_names.index(name) % len(palette)]

# ---- 训练集前 6 张图的真实边界框（GT）----
# rois 里是归一化角坐标，画回原图要乘 bbox_scale（书 7.1 节的做法）
fig, axes = d2l.plt.subplots(2, 3, figsize=(15, 10))
for ax, (_, row) in zip(axes.ravel(), train_data.head(6).iterrows()):
    ax.imshow(Image.open(row['image']))
    ax.axis('off')
    bbox_scale = torch.tensor((row['width'], row['height'],
                               row['width'], row['height']))
    boxes = torch.tensor([[r['xmin'], r['ymin'], r['xmax'], r['ymax']]
                          for r in row['rois']]) * bbox_scale
    names = [r['class'] for r in row['rois']]
    d2l.show_bboxes(ax, boxes, names, [cls_color(n) for n in names])
d2l.plt.show()

# ---- 验证集前 3 张图的模型预测结果：直观对比漏检/误检 ----
fig, axes = d2l.plt.subplots(1, 3, figsize=(15, 5))
preds = predict_paths(model, valid_data['image'].head(3).tolist(), tta=False)
for ax, (pred, (_, row)) in zip(axes, zip(preds, valid_data.head(3).iterrows())):
    ax.imshow(Image.open(row['image']))
    ax.axis('off')
    names = [class_names[int(c)] for c in pred.boxes.cls.tolist()]
    labels = [f'{n} {s:.2f}' for n, s in zip(names, pred.boxes.conf.tolist())]
    d2l.show_bboxes(ax, pred.boxes.xyxy.cpu(), labels, [cls_color(n) for n in names])
d2l.plt.show()

## 小结

* 迁移学习让小数据集检测成为可能：COCO 预训练的 YOLOv5su 只需按目标类别数重建检测头，其余权重全部微调。
* 类别索引有两套：YOLO 内部是 0~N-1，COCO/比赛提交用任意整数 id——预测输出必须查表换回原 id。
* 数据层面应对类别不平衡：按类别分层切分验证集保证稀有类评估可信；对极端稀有的 belt 做图像级过采样（复制图片列表行 + 随机增强）。
* 高分辨率输入（imgsz=1024）是小目标 AP 的第一杠杆；mosaic 增强和 TTA 是几乎白拿的提升。
* 评估用 COCO 官方 COCOeval，与两阶段检测器（Faster R-CNN）的旧结果直接可比。